In [5]:
import pandas as pd

# pip install -U pandas pyarrow
df = pd.read_parquet("/projectnb/rlhf/mingyuc/verl_github/verl/data/multiturn/train50000.parquet")   # 若是 S3 路径同样支持


In [2]:
print(len(df))

10000


In [6]:

chat_list = [df['messages'][i] for i in range(len(df))]

In [18]:
from vllm import LLM, SamplingParams

#del llm

# 替换为你的本地模型路径

#local_model_path = "/projectnb/rlhf/mingyuc/DisCO/exp/qwen-1.5b/global_step_665"
#local_model_path = "/projectnb/rlhf/mingyuc/verl_github/verl/sft/global_step_312"

#local_model_path = "/projectnb/rlhf/mingyuc/exp/qwen1.5/global_step_78"

#local_model_path = "Qwen/Qwen2.5-1.5B"

#llm = LLM(model=local_model_path)

: 

In [17]:
sampling_params = SamplingParams(temperature=0.6, max_tokens=6)
tokenizer = llm.get_tokenizer()

index = 2


chat_exp = chat_list[5]
print(len(chat_exp))
test_chat = chat_exp[:index]
chat_str = tokenizer.apply_chat_template(test_chat, add_generation_prompt=True, tokenize=False)
output = llm.generate(chat_str, sampling_params)
print(output[0].outputs[0].text)
print(chat_exp[index-1:index+1])

result = chat_exp[index]['content'] in output[0].outputs[0].text
print(result)



375


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 20.57it/s, est. speed input: 2597.21 toks/s, output: 123.63 toks/s]

(7, 4)
[{'content': 'Trajectory 1: (7, 4): path, (7, 6): path, (6, 5): wall, (8, 5): wall', 'role': 'user'}
 {'content': '(7, 4)', 'role': 'assistant'}]
True


In [6]:
sampling_params = SamplingParams(temperature=0.0, max_tokens=6)
tokenizer = llm.get_tokenizer()
chat_strs = []
targets = []

for chat_exp in chat_list[:100]:
    for index in range(1, len(chat_exp), 2):
        test_chat = chat_exp[:index]
        chat_str = tokenizer.apply_chat_template(test_chat, add_generation_prompt=True, tokenize=False)
        chat_str = chat_str.replace(
            "<|im_start|>system\nYou are a helpful assistant.<|im_end|>",
            "<|im_start|>system\nYou are an intelligent agent navigating a maze.\nAt each step, you receive an observation of four adjacent cells, described by their coordinates and whether they are 'path' or 'wall'.\nYou must choose exactly one adjacent cell that is a valid 'path' or 'exit' and move into it.\nAlways move efficiently toward the goal.\nOutput your next move as a single coordinate in the format (row, col).\nDo not explain or repeat the input — just return the next move.\n<|im_end|>"
        )
        chat_strs.append(chat_str)
        targets.append(chat_exp[index]['content'])

# 并行推理
outputs = llm.generate(chat_strs, sampling_params)

# 统计正确率
total = len(outputs)
correct = 0
for i, output in enumerate(outputs):
    result = targets[i] in output.outputs[0].text
    if result:
        correct += 1

accuracy = correct / total if total > 0 else 0
print(f"总数: {total}, 正确数: {correct}, 正确率: {accuracy:.4f}")

Processed prompts: 100%|██████████| 3234/3234 [01:18<00:00, 41.36it/s, est. speed input: 40603.79 toks/s, output: 248.17 toks/s]

总数: 3234, 正确数: 3048, 正确率: 0.9425


In [7]:
sampling_params = SamplingParams(temperature=0.0, max_tokens=6)
tokenizer = llm.get_tokenizer()
chat_strs = []
targets = []

for chat_exp in chat_list[:100]:
    for index in range(1, len(chat_exp), 2):
        test_chat = chat_exp[:index]
        chat_str = tokenizer.apply_chat_template(test_chat, add_generation_prompt=True, tokenize=False)
        chat_str = chat_str.replace(
            "<|im_start|>system\nYou are a helpful assistant.<|im_end|>",
            "<|im_start|>system\nYou are an intelligent agent navigating a maze.\nAt each step, you receive an observation of four adjacent cells, described by their coordinates and whether they are 'path' or 'wall'.\nYou must choose exactly one adjacent cell that is a valid 'path' or 'exit' and move into it.\nAlways move efficiently toward the goal.\nOutput your next move as a single coordinate in the format (row, col).\nDo not explain or repeat the input — just return the next move.\n<|im_end|>"
        )
        chat_strs.append(chat_str)
        targets.append(chat_exp[index]['content'])

In [12]:
import pandas as pd

data = []
for chat_exp in chat_list[:5000]:
    for index in range(len(chat_exp)):
        if chat_exp[index]['role'] != 'assistant':
            continue
        test_chat = chat_exp[:index]
        answer = chat_exp[index]['content']
        data.append({
            "extra_info": {
                "question": test_chat,
                "answer": answer
            }
        })

df = pd.DataFrame(data)

In [13]:
print(len(df)   )

368926


In [14]:
print(data[124])

{'extra_info': {'question': array([{'content': "You are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only.", 'role': 'system'},
       {'content': 'Trajectory 1: (1, 6): wall, (1, 8): wall, (0, 7): wall, (2, 7): path', 'role': 'user'},
       {'content': '(2, 7)', 'role': 'assistant'},
       {'content': 'Trajectory 1: (2, 6): wall, (2, 8): wall, (1, 7): path, (3, 7): path', 'role': 'user'},
       {'content': '(3, 7)', 'role': 'assistant'},
       {'content': 'Trajectory 1: (3, 6): path, (3, 8): wall, (2, 7): path, (4, 7): wall', 'role': 'user'},
       {'content': '(3, 6)', 'role': 'assistant'},
       {'content': 'Trajectory 1: (3, 5): path, (3, 7): path, (2, 6): wall, (4

In [11]:
import datasets
from datasets import Dataset
df.to_parquet('/projectnb/rlhf/mingyuc/DisCO/datasets/maze/train_one_tune.parquet',
              engine='pyarrow',           # 推荐，用 Arrow 写更快
              index=False)                # 不把行索引写进去

In [1]:
import pandas as pd 
df_mt = pd.read_parquet("/projectnb/rlhf/mingyuc/verl_github/verl/data/maze_mt/train2000.parquet")


In [5]:
# Inspect df_mt structure and columns
print("df_mt shape:", df_mt.shape)
print("df_mt columns:", df_mt.columns.tolist())
print("\nFirst few rows:")
print(df_mt.head())

# Check if there's a messages or extra_info column
if 'messages' in df_mt.columns:
    print("\n'messages' column sample:")
    print(df_mt['messages'].iloc[0])
elif 'extra_info' in df_mt.columns:
    print("\n'extra_info' column sample:")
    print(df_mt['extra_info'].iloc[0])
else:
    print("\nNo 'messages' or 'extra_info' column found. Available columns:")
    for col in df_mt.columns:
        print(f"  {col}: {type(df_mt[col].iloc[0])}")
        if hasattr(df_mt[col].iloc[0], '__len__') and len(str(df_mt[col].iloc[0])) < 200:
            print(f"    Sample: {df_mt[col].iloc[0]}")
        else:
            print(f"    Sample: {str(df_mt[col].iloc[0])[:100]}...")

df_mt shape: (2000, 1)
df_mt columns: ['extra_info']

First few rows:
                                          extra_info
0  {'chat': [{'content': 'Trajectory 1: (1, 2): w...
1  {'chat': [{'content': 'Trajectory 1: (5, 0): w...
2  {'chat': [{'content': 'Trajectory 1: (3, 2): p...
3  {'chat': [{'content': 'Trajectory 1: (7, 6): p...
4  {'chat': [{'content': 'Trajectory 1: (5, 6): p...

'extra_info' column sample:
{'chat': array([{'content': 'Trajectory 1: (1, 2): wall, (1, 4): path, (0, 3): wall, (2, 3): path', 'role': 'user'},
       {'content': '(1, 4)', 'role': 'assistant'},
       {'content': 'Trajectory 1: (1, 3): path, (1, 5): path, (0, 4): wall, (2, 4): wall', 'role': 'user'},
       {'content': '(1, 5)', 'role': 'assistant'},
       {'content': 'Trajectory 1: (1, 4): path, (1, 6): path, (0, 5): wall, (2, 5): path', 'role': 'user'},
       {'content': '(1, 6)', 'role': 'assistant'},
       {'content': 'Trajectory 1: (1, 5): path, (1, 7): path, (0, 6): wall, (2, 6): wall', 'role'

In [6]:
# Inspect the structure of df_mt to understand how to extract chat data
print("df_mt columns:", df_mt.columns.tolist())
print("df_mt shape:", df_mt.shape)
print("\nFirst row preview:")
print(df_mt.iloc[0])

# Check if there's an 'extra_info' column with 'chat' field
if 'extra_info' in df_mt.columns:

    sample_extra_info = df_mt['extra_info'].iloc[0]
    print("Type:", type(sample_extra_info))
    if isinstance(sample_extra_info, dict):
        print("Keys:", sample_extra_info.keys())
        if 'chat' in sample_extra_info:
            print("Found 'chat' in extra_info")
            print("Chat type:", type(sample_extra_info['chat']))
            if isinstance(sample_extra_info['chat'], list) and len(sample_extra_info['chat']) > 0:
                print("First message:", sample_extra_info['chat'][0])

df_mt columns: ['extra_info']
df_mt shape: (2000, 1)

First row preview:
extra_info    {'chat': [{'content': 'Trajectory 1: (1, 2): w...
Name: 0, dtype: object
Type: <class 'dict'>
Keys: dict_keys(['chat'])
Found 'chat' in extra_info
Chat type: <class 'numpy.ndarray'>


In [9]:
# First extract chat_list_mt from df_mt 
# Check the structure and extract chat data similar to how chat_list was extracted above
if 'extra_info' in df_mt.columns:
    chat_list_mt = df_mt['extra_info'].apply(lambda x: x['chat'] if isinstance(x, dict) and 'chat' in x else []).tolist()
elif 'messages' in df_mt.columns:
    chat_list_mt = df_mt['messages'].tolist()
else:
    # If the structure is different, try to find the chat data in other columns
    print("Available columns in df_mt:", df_mt.columns.tolist())
    # You may need to adjust this based on the actual structure
    chat_list_mt = []



# Convert maze_mt data to multiturn.py conversation format
conversations = []

for chat in chat_list_mt:
    # Create a conversation dictionary with "messages" key
    conversation = {"messages": []}
    
    # Add messages to the conversation
    for message in chat:
        if isinstance(message, dict) and 'role' in message and 'content' in message:
            conversation["messages"].append({
                "role": message["role"],
                "content": message["content"]
            })
    
    # Only add conversations that have at least one message
    if conversation["messages"]:
        conversations.append(conversation)

print(f"Converted {len(conversations)} conversations to multiturn format")
print(f"First conversation preview:")
if conversations:
    print(f"Number of messages: {len(conversations[0]['messages'])}")
    for i, msg in enumerate(conversations[0]['messages'][:3]):  # Show first 3 messages
        print(f"  Message {i+1} ({msg['role']}): {msg['content'][:100]}...")

Converted 2000 conversations to multiturn format
First conversation preview:
Number of messages: 282
  Message 1 (user): Trajectory 1: (1, 2): wall, (1, 4): path, (0, 3): wall, (2, 3): path...
  Message 2 (assistant): (1, 4)...
  Message 3 (user): Trajectory 1: (1, 3): path, (1, 5): path, (0, 4): wall, (2, 4): wall...


In [12]:
print(conversations[1])

{'messages': [{'role': 'user', 'content': 'Trajectory 1: (5, 0): wall, (5, 2): path, (4, 1): path, (6, 1): path'}, {'role': 'assistant', 'content': '(5, 2)'}, {'role': 'user', 'content': 'Trajectory 1: (5, 1): path, (5, 3): path, (4, 2): wall, (6, 2): wall'}, {'role': 'assistant', 'content': '(5, 3)'}, {'role': 'user', 'content': 'Trajectory 1: (5, 2): path, (5, 4): wall, (4, 3): wall, (6, 3): wall'}, {'role': 'assistant', 'content': '(5, 2)'}, {'role': 'user', 'content': 'Trajectory 1: (5, 1): path, (5, 3): path, (4, 2): wall, (6, 2): wall'}, {'role': 'assistant', 'content': '(5, 1)'}, {'role': 'user', 'content': 'Trajectory 1: (5, 0): wall, (5, 2): path, (4, 1): path, (6, 1): path'}, {'role': 'assistant', 'content': '(4, 1)'}, {'role': 'user', 'content': 'Trajectory 1: (4, 0): wall, (4, 2): wall, (3, 1): path, (5, 1): path'}, {'role': 'assistant', 'content': '(5, 1)'}, {'role': 'user', 'content': 'Trajectory 1: (5, 0): wall, (5, 2): path, (4, 1): path, (6, 1): path'}, {'role': 'assis